PREPROCESSING

In [1]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pydub import AudioSegment
from pathlib import Path
from datetime import datetime
import traceback
from scipy.signal import butter, sosfilt
import soundfile as sf
import pandas as pd
import os
import matplotlib.dates as mdates
import sys
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import noisereduce as nr
import shutil

c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=sys.maxsize)

INPUT_DIR = Path("AudioSamplesFiltered")
#CSV_INPUT_PATH = Path("audio_samples_metadata.csv")
NOISE_PATH = Path("noise.wav")
OUTPUT_DIR_PREPROCESSED = Path("AudioSamplesPreprocessed")
#OUTPUT_DIR_PREPROCESSED_NORMALIZED = Path("AudioSamplesPreprocessedNormalized")
#OUTPUT_DIR_PREPROCESSED_NORMALIZED.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000
FRAME_SIZE = 2048
HOP_LENGTH = 512
TARGET_DB = -1.0
target_amplitude = librosa.db_to_amplitude(TARGET_DB)
CUTOFF_FREQ = 100
NOISE_REDUCTION_PROPORTION = 1.0


In [3]:
if OUTPUT_DIR_PREPROCESSED.exists():
    shutil.rmtree(OUTPUT_DIR_PREPROCESSED)  # Rimuove la cartella e tutto il contenuto

OUTPUT_DIR_PREPROCESSED.mkdir(parents=True) # La ricrea vuota

In [4]:
def highpass_filter(data, cutoff, sr, order=5):
    # Nyquist frequency è la metà della frequenza di campionamento
    nyq = 0.5 * sr
    normal_cutoff = cutoff / nyq
    
    # Crea il filtro in formato SOS (Second-Order Sections) 
    # È più stabile numericamente rispetto al formato classico
    sos = butter(order, normal_cutoff, btype='high', analog=False, output='sos')
    
    # Applica il filtro
    filtered_data = sosfilt(sos, data)
    return filtered_data

def preprocess(audio, offset_to_cut=0.5, normalize=False, noise_signal=None):
    # 1. RIMOZIONE COMPONENTE DC 
    audio = audio - np.mean(audio)

    # 2. TAGLIO
    offset = int(offset_to_cut * SAMPLE_RATE)
    audio = audio[offset:-offset]

    if noise_signal is not None:
        noise_signal = noise_signal - np.mean(noise_signal)
        noise_signal =  noise_signal[offset:-offset]
        audio = nr.reduce_noise(y=audio, sr=SAMPLE_RATE, y_noise=noise_signal, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH, prop_decrease=NOISE_REDUCTION_PROPORTION, stationary=False)
    

    #audio = highpass_filter(audio, cutoff=CUTOFF_FREQ, sr=SAMPLE_RATE, order=8)
    if normalize:
        audio = librosa.util.normalize(audio) * target_amplitude
    return audio

In [5]:
audio_files = list(INPUT_DIR.glob("*.wav"))

total_files = len(audio_files)

In [6]:
noise, _ = librosa.load(NOISE_PATH, sr=SAMPLE_RATE)

# 1. RIMOZIONE COMPONENTE DC 
#noise = noise - np.mean(noise)

In [7]:
for i, filepath in enumerate(audio_files, 1):
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE)

    preprocessed_audio=preprocess(audio, offset_to_cut=0.5, normalize=False, noise_signal=noise)
    #normalized_preprocessed_audio=preprocess(audio, offset_to_cut=0.5, normalize=True, noise_signal=noise)
    
    output_path = OUTPUT_DIR_PREPROCESSED / filepath.name
    #normalized_output_path = OUTPUT_DIR_PREPROCESSED_NORMALIZED / filepath.name

    sf.write(output_path, preprocessed_audio, SAMPLE_RATE)
    #sf.write(normalized_output_path, normalized_preprocessed_audio, SAMPLE_RATE)

    print(f"[{i}/{total_files}] Salvato: {output_path}")
    

[1/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-26-59.wav
[2/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-28-11.wav
[3/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T10-37-03.wav
[4/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T10-38-15.wav
[5/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T10-39-26.wav
[6/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T10-40-37.wav
[7/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T10-53-07.wav
[8/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T10-54-18.wav
[9/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T10-56-41.wav
[10/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T11-10-22.wav
[11/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T11-11-34.wav
[12/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T11-12-45.wav
[13/2554] Salvato: AudioSamplesPreprocessed\audio_2026-02-23T11-13-56.wav
[14/2554] Salvato: AudioSamplesPreprocessed\aud